# Beyond Rank Fusion: A Practical Guide to Elasticsearch Retrievers

---
## Install Prerequisites

In [1]:
! pip install -q -U -r requirements.txt

---
## Provision Elastic Serverless (Terraform)

In [ ]:
%%bash
terraform -chdir=terraform init -upgrade
terraform -chdir=terraform apply -auto-approve

---
## Create .env from Terraform Outputs

In [ ]:
%%bash
cat > .env << EOF
ELASTIC_USERNAME=$(terraform -chdir=terraform output -raw elastic_username)
ELASTIC_PASSWORD=$(terraform -chdir=terraform output -raw elastic_password)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
EOF

---
## Section 1 — Environment Setup

Connect to the Elastic cluster and load the product catalog.

In [ ]:
import json
import os

from dotenv import load_dotenv
from elasticsearch import Elasticsearch
import pandas as pd

from lib.helpers import create_index, ingest_products, display_results, side_by_side

load_dotenv(override=True)

client = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    basic_auth=(os.environ["ELASTIC_USERNAME"], os.environ["ELASTIC_PASSWORD"]),
)

with open("data/products.json") as f:
    products = json.load(f)

print(f"Loaded {len(products)} products")

create_index(client)
ingest_products(client, products)
client.indices.refresh(index="hand_tools")
count = client.count(index="hand_tools")
print(f"Document count: {count['count']}")

---
## Section 2 — Baseline: BM25 vs Semantic

Neither BM25 nor semantic search alone gives the full picture. Let's see how they differ on the query: *"versatile fastening tool for woodworking joints"*

In [ ]:
# BM25 keyword search
QUERY = "versatile fastening tool for woodworking joints"

bm25_resp = client.search(
    index="hand_tools",
    body={
        "query": {
            "multi_match": {
                "query": QUERY,
                "fields": ["name", "description"],
            }
        },
        "size": 5,
    },
)

bm25_df = display_results(bm25_resp["hits"]["hits"])
print("BM25 Keyword Search Results:")
display(bm25_df)

# Semantic search via EIS
semantic_resp = client.search(
    index="hand_tools",
    body={
        "query": {
            "semantic": {
                "field": "description_semantic",
                "query": QUERY,
            }
        },
        "size": 5,
    },
)

semantic_df = display_results(semantic_resp["hits"]["hits"])
print("\nSemantic Search Results:")
display(semantic_df)

> **Observation:** BM25 rewards exact keyword overlap — "fastening" and "woodworking" score well. The semantic query uses Jina Embeddings via EIS to find meaning beyond the words — it may surface products that never use the word "fastening" but are conceptually relevant. Neither is wrong. The challenge is combining them intelligently.

---
## Section 3 — RRF Retriever (Reciprocal Rank Fusion)

RRF merges ranked lists from multiple retrievers using rank positions only — it never looks at the actual scores. We start with equal weights, then show how per-retriever weights shift rankings.

**Query:** *"precision screwdriver for electronics repair"*

In [ ]:
# Unweighted RRF
QUERY = "precision screwdriver for electronics repair"

rrf_resp = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        }
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        }
                    },
                ],
                "rank_window_size": 50,
            }
        },

        "size": 7,
    },
)

rrf_df = display_results(rrf_resp["hits"]["hits"])
rrf_df

> **Look for Trap A1** (“PrecisionDrive 6-Piece”) and **Trap A2** (“Vessel Impacta JIS Driver”). A1 is keyword-stuffed — its description repeats “precision screwdriver” without adding real product detail. A1 has been gamed to rank well on *both* BM25 and semantic legs, so RRF keeps it near the top.
>
> A2 is the opposite: its description richly describes electronics repair work (circuit boards, soldering, cramped enclosures) without using the query’s keywords. This makes it semantically strong but nearly invisible to BM25 — and since RRF averages ranks from both legs, A2 gets dragged down or drops out entirely.
>
> **The lesson:** hybrid search does not automatically neutralise keyword stuffing. Rank fusion alone is not enough — you need score-aware fusion (Linear, Section 5) and business-signal rescoring (Rescorer, Section 6) to fix this.

In [ ]:
# Weighted RRF — boost BM25 retriever (weight: 2.0)
rrf_bm25_boost = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 2.0,
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        }
                    }
                ],
                "rank_constant": 60,
                "rank_window_size": 50,
            }
        },
        "size": 7,
    },
)

rrf_bm25_df = display_results(rrf_bm25_boost["hits"]["hits"])
rrf_bm25_df

In [ ]:
# Weighted RRF — boost semantic retriever (weight: 2.0)
rrf_sem_boost = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        }      
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 2.0,
                    }
                ],
                "rank_constant": 60,
                "rank_window_size": 50,
            }
        },
        "size": 7,
    },
)

rrf_sem_df = display_results(rrf_sem_boost["hits"]["hits"])
rrf_sem_df

The table below compares three RRF variants side-by-side, sorted by the unweighted RRF score. A dash (—) means the product did not appear in that variant's top results.

Key things to look for:
- **A1 (PrecisionDrive)** sits at #1 across all three columns — keyword stuffing games both retrievers, so no amount of RRF reweighting dislodges it.
- **A2 (Vessel Impacta)** is absent from unweighted RRF and BM25-boosted RRF entirely. It only appears when semantic is boosted — and even then it's dead last (#7), still below A1.
- **A3 (Wiha)** — the genuinely relevant product — is #2 in every column but can never overtake the keyword-stuffed A1.
- **Mitutoyo** (a caliper, not a screwdriver) appears in unweighted and BM25-boosted RRF but drops out under semantic boost — semantic search correctly identifies it as irrelevant.
- **Knipex** (pliers) appears in all three columns, hovering near the bottom — it's marginally relevant on both legs, so reweighting doesn't remove it.

In [ ]:
# Side-by-side: unweighted vs BM25-boosted vs semantic-boosted
side_by_side({"rrf": rrf_df, "rrf_bm25_boost": rrf_bm25_df, "rrf_sem_boost": rrf_sem_df})

> **Takeaway:** Weighted RRF gives you a dial — but the dial only adjusts how much each retriever’s *rank position* counts. Boosting semantic weight surfaces A2 and drops irrelevant products like Mitutoyo, but A1 stays at #1 regardless of weighting because it games both legs. RRF has no way to penalise shallow, keyword-stuffed content — it cannot tell whether rank 1 was a near-perfect match or barely squeaked past rank 2. We need score magnitudes and business signals.

---
## Section 4 — Linear Retriever

Linear retriever combines actual score values instead of just ranks. With MinMax normalization, scores from different retrievers are mapped to a common 0-1 scale before weighting.

**Query:** *"mortise chisel for hand-cut joinery"*

In [ ]:
# RRF baseline for this query
QUERY = "mortise chisel for hand-cut joinery"

rrf_baseline = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "standard": {
                            "query": {
                                "multi_match": {
                                    "query": QUERY,
                                    "fields": ["name", "description"],
                                }
                            }
                        }
                    },
                    {
                        "standard": {
                            "query": {
                                "semantic": {
                                    "field": "description_semantic",
                                    "query": QUERY,
                                }
                            }
                        }
                    },
                ],
                "rank_constant": 60,
                "rank_window_size": 50,
            }
        },
        "size": 7,
    },
)

rrf_base_df = display_results(rrf_baseline["hits"]["hits"])
rrf_base_df

In [ ]:
# Linear retriever — MinMax normalization, BM25 0.3 / semantic 0.7 
linear_swap = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.3,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.7,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

linear_swap_df = display_results(linear_swap["hits"]["hits"])
linear_swap_df

In [ ]:
# Linear retriever — MinMax normalization, BM25 0.6 / semantic 0.4
linear_minmax = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.6,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.4,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

linear_mm_df = display_results(linear_minmax["hits"]["hits"])
linear_mm_df

The table below compares RRF (rank-based) against two Linear retrievers (score-based) with different weight splits. All scores are MinMax-normalised to 0–1 for the Linear columns.

Key things to look for:
- **Mortise Chisel Heavy-Duty** scores 1.0 in both Linear columns — it dominates both BM25 and semantic by such a margin that MinMax maps it to the ceiling. RRF can't express this dominance; it just sees "rank 1."
- **Two Cherries Pigsticker** appears in both Linear columns but not in RRF. Linear's score awareness surfaces it; RRF's rank cutoff drops it.
- **Two Cherries Firmer Chisel** is the reverse — it appears in RRF but not in either Linear column. It ranked well enough on both legs for RRF but its actual scores were too low for Linear to keep it.
- **Narex vs Pigsticker**: under BM25-heavy weights (0.6), Pigsticker (0.20) is competitive with Narex (0.27). Shift to semantic-heavy weights (0.7) and Pigsticker collapses to 0.10 while Narex rises to 0.30 — the gap triples. Pigsticker's relevance was propped up by BM25 keyword matching; semantic search sees through it.
- **Gyokucho** (a saw, not a chisel) drops from 0.10 → 0.09 when semantic weight increases — semantic search penalises it slightly more than BM25 does.

In [ ]:
# Side-by-side: RRF vs Linear (BM25-heavy) vs Linear (semantic-heavy)
side_by_side({"rrf": rrf_base_df, "linear_0.6_bm25": linear_mm_df, "linear_0.7_sem": linear_swap_df})

> **Takeaway:** Linear sees the *magnitude* of scores, not just their rank. When the Mortise Chisel scores 52 on BM25 and the next product scores 11, Linear respects that gap — RRF does not. The weight dial now has real teeth: shifting from BM25-heavy to semantic-heavy collapsed the Pigsticker’s score by half while lifting Narex, exposing which products have genuine semantic relevance versus keyword-only relevance. But Linear still operates purely on retrieval scores — it has no idea what `avg_rating` or `units_sold_30d` are, and it cannot penalise out-of-stock products or promote bestsellers. For that, we need the Rescorer.

---
## Section 5 — Rescorer Retriever

The Rescorer wraps a first-stage retriever (here, Linear) and applies a script score that can incorporate business signals like ratings, sales velocity, and stock status.

**Query:** *"reliable claw hammer for general carpentry"*

In [ ]:
# Linear baseline (no rescoring)
QUERY = "reliable claw hammer for general carpentry"

linear_base = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.6,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.4,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

linear_base_df = display_results(linear_base["hits"]["hits"])
linear_base_df

In [ ]:
# Rescorer — boost by sales velocity and rating
rescorer_resp = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rescorer": {
                "retriever": {
                    "linear": {
                        "rank_window_size": 50,
                        "retrievers": [
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "multi_match": {
                                                "query": QUERY,
                                                "fields": ["name", "description"],
                                            }
                                        }
                                    }
                                },
                                "weight": 0.6,
                                "normalizer": "minmax",
                            },
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "semantic": {
                                                "field": "description_semantic",
                                                "query": QUERY,
                                            }
                                        }
                                    }
                                },
                                "weight": 0.4,
                                "normalizer": "minmax",
                            },
                        ],
                    }
                },
                "rescore": {
                    "window_size": 50,
                    "query": {
                        "rescore_query": {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "_score * Math.log1p(doc['units_sold_30d'].value) * (doc['avg_rating'].value / 5.0)"
                                },
                            }
                        },
                        "query_weight": 1,
                        "rescore_query_weight": 2,
                    },
                },
            }
        },
        "size": 7,
    },
)

rescorer_df = display_results(rescorer_resp["hits"]["hits"])
rescorer_df

In [ ]:
# Rescorer + in_stock penalty
rescorer_stock = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rescorer": {
                "retriever": {
                    "linear": {
                        "rank_window_size": 50,
                        "retrievers": [
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "multi_match": {
                                                "query": QUERY,
                                                "fields": ["name", "description"],
                                            }
                                        }
                                    }
                                },
                                "weight": 0.6,
                                "normalizer": "minmax",
                            },
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "semantic": {
                                                "field": "description_semantic",
                                                "query": QUERY,
                                            }
                                        }
                                    }
                                },
                                "weight": 0.4,
                                "normalizer": "minmax",
                            },
                        ],
                    }
                },
                "rescore": {
                    "window_size": 50,
                    "query": {
                        "rescore_query": {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "_score * Math.log1p(doc['units_sold_30d'].value) * (doc['avg_rating'].value / 5.0) * (doc['in_stock'].value ? 1.0 : 0.1)"
                                },
                            }
                        },
                        "query_weight": 1,
                        "rescore_query_weight": 2,
                    },
                },
            }
        },
        "size": 7,
    },
)

rescorer_stock_df = display_results(rescorer_stock["hits"]["hits"])
rescorer_stock_df

In [ ]:
# Score breakdown — show the math behind rescoring
# final = query_weight * linear_score + rescore_query_weight * rescore_score
# With query_weight=1, rescore_query_weight=2:
#   final = linear_score + 2 * (linear_score * log1p(units_sold_30d) * avg_rating/5)

breakdown_resp = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.6,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.4,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

import math
rows = []
for hit in breakdown_resp["hits"]["hits"]:
    src = hit["_source"]
    linear_score = hit["_score"]
    rating = src["avg_rating"]
    sold = src["units_sold_30d"]
    in_stock = src["in_stock"]

    rating_mult = rating / 5.0
    velocity_mult = math.log1p(sold)
    stock_mult = 1.0 if in_stock else 0.1
    rescore_score = linear_score * velocity_mult * rating_mult
    final = linear_score + 2 * rescore_score
    final_with_stock = linear_score + 2 * (rescore_score * stock_mult)

    rows.append({
        "name": src["name"],
        "linear_score": round(linear_score, 4),
        "avg_rating": rating,
        "units_sold_30d": sold,
        "in_stock": in_stock,
        "rating_mult": round(rating_mult, 2),
        "velocity_mult": round(velocity_mult, 2),
        "stock_mult": stock_mult,
        "rescored": round(final, 4),
        "rescored_stock": round(final_with_stock, 4),
    })

pd.DataFrame(rows)

In [ ]:
# Side-by-side: Linear vs Rescorer vs Rescorer+stock
side_by_side({"linear": linear_base_df, "rescorer": rescorer_df, "rescorer_stock": rescorer_stock_df})

**Reading guide — what to look for row by row:**

| Product | `linear_score` | `rescorer_score` | `rescorer_stock_score` | What happened |
|---------|---------------|-----------------|----------------------|---------------|
| Reliable Claw Hammer | 0.98 (rank 1) | — | — | Best text match but `avg_rating` 1.9 and `units_sold_30d` 14 → business signals are so weak the rescorer drops it out of the top 7 entirely |
| **Lie-Nielsen Hammer** | **0.84 (rank 2)** | **11.87** | **—** | Strong text match *and* strong business signals (rating 4.8, 310 sold) push it to rescorer #2 — but `in_stock: false` triggers the stock penalty, removing it from rescorer+stock |
| Estwing E3-16C | 0.07 | 14.88 (rank 1) | 14.88 (rank 1) | Low text relevance but the best business signals in the set (rating 4.9, 1840 sold) → rescorer vaults it to #1, and it stays #1 with stock filter since it's in stock |
| Knipex, Klein, Irwin, Wera | — | ~10.9–11.4 | ~10.9–11.4 | Not in the linear top 7 at all, but the rescorer's `window_size: 50` rescores all 50 candidates — these products have strong business signals that overcome their low relevance |

The **Lie-Nielsen row** is the key demonstration: a product can rank well on relevance *and* business signals, yet still be excluded by a single boolean filter. This is the kind of business logic that neither RRF nor Linear can express.

> **Takeaway:** The Rescorer doesn't change what gets *retrieved* — it changes what gets *promoted*. The candidate set comes from your best hybrid retriever. The Rescorer then layers on business logic that no query can express. Notice the **Lie-Nielsen Warrington Pattern Hammer**: it ranks #2 in the linear baseline and earns a strong rescorer score (11.87) thanks to its high `avg_rating` (4.8) and 310 units sold — but it vanishes from the `rescorer_stock` column because `in_stock` is `false`. One boolean field overrides all the relevance and popularity signals. That is the power of the Rescorer: it lets you inject hard business constraints *after* fusion, so out-of-stock products never reach the customer no matter how relevant they are.

---
## Section 6 — Summary & Comparison

| Retriever          | Fusion method       | Score magnitude? | Weights? | Post-retrieval business logic? |
|--------------------|---------------------|------------------|----------|-------------------------------|
| RRF (unweighted)   | Rank-based          | No               | No       | No                            |
| RRF (weighted)     | Rank-based          | No               | Yes      | No                            |
| Linear + MinMax    | Score-based         | Yes              | Yes      | No                            |
| Rescorer           | Score-based + rerank| Yes              | Yes      | **Yes**                       |

In [ ]:
# All four retrievers on a single query
QUERY = "durable hand tool for precise woodworking"

# 1. RRF (unweighted)
r1 = client.search(index="hand_tools", body={
    "retriever": {"rrf": {"retrievers": [
        {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}},
        {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}},
    ], "rank_constant": 60, "rank_window_size": 50}}, "size": 5})

# 2. RRF (weighted — BM25 boosted)
r2 = client.search(index="hand_tools", body={
    "retriever": {"rrf": {"retrievers": [
        {"retriever": {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}}, "weight": 2.0},
        {"retriever": {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}}},
    ], "rank_constant": 60, "rank_window_size": 50}}, "size": 5})

# 3. Linear + MinMax
r3 = client.search(index="hand_tools", body={
    "retriever": {"linear": {"retrievers": [
        {"retriever": {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}}, "weight": 0.6, "normalizer": "minmax"},
        {"retriever": {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}}, "weight": 0.4, "normalizer": "minmax"},
    ]}}, "size": 5})

# 4. Rescorer (relevance + business signals, same candidate pool as Linear)
r4 = client.search(index="hand_tools", body={
    "retriever": {"rescorer": {
        "retriever": {"linear": {"retrievers": [
            {"retriever": {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}}, "weight": 0.6, "normalizer": "minmax"},
            {"retriever": {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}}, "weight": 0.4, "normalizer": "minmax"},
        ], "rank_window_size": 5}},
        "rescore": {"window_size": 5, "query": {
            "rescore_query": {"script_score": {"query": {"match_all": {}},
                "script": {"source": "_score * Math.log1p(doc['units_sold_30d'].value) * (doc['avg_rating'].value / 5.0)"}}},
            "query_weight": 1, "rescore_query_weight": 2}},
    }}, "size": 5})

frames = {
    "rrf": display_results(r1["hits"]["hits"]),
    "rrf_weighted": display_results(r2["hits"]["hits"]),
    "linear_minmax": display_results(r3["hits"]["hits"]),
    "rescorer": display_results(r4["hits"]["hits"]),
}

side_by_side(frames)

**Reading guide:**

All four retrievers run the same query. A "—" means the product didn't appear in that retriever's top 5.

- **RRF and Linear** agree on the same relevant products — saws, vises, chisels, drills — but rank them differently. RRF uses rank positions; Linear uses normalized score magnitudes.
- **Rescorer** takes the Linear top 5 and re-orders them by blending relevance with business signals. Notice the rank changes:
  - **Record Vise** jumps from Linear #3 (0.40) to Rescorer #1 (10.25) — modest relevance but strong sales and ratings.
  - **Gyokucho Saw** stays near the top in both (Linear #1 → Rescorer #2) — high relevance *and* solid business signals.
  - **Schroeder Drill** drops from Linear #4 to Rescorer #5 — decent relevance but weaker business metrics pull it down.
- **PrecisionDrive** (the keyword-stuffed trap) sneaks into Linear and Rescorer but is absent from both RRF variants — rank-based fusion filters it out.
- **Mortise Chisel** appears in both RRF variants but not in Linear or Rescorer — the BM25 and semantic retrievers rank it just differently enough that RRF's rank fusion promotes it, while Linear's score-based fusion does not.

### When to use each retriever

- **RRF (unweighted):** Good starting point when you have multiple retrieval signals and no prior knowledge about which matters more. Zero tuning required.
- **RRF (weighted):** When you know one retrieval method is more reliable for your domain — e.g., boost semantic weight for natural-language queries, boost BM25 for part-number lookups.
- **Linear + MinMax:** When score magnitude matters — products that are dramatically more relevant should rank dramatically higher, not just one rank position above. Requires some weight tuning.
- **Rescorer:** When business logic must influence final rankings — popularity, ratings, stock status, recency, margin, or any field-level signal that pure text relevance cannot capture.

---
## Teardown — Destroy Elastic Environment

In [ ]:
%%bash
terraform -chdir=terraform destroy -auto-approve
rm -f .env